# Alpha and Signal-Noise Sweeps

Follow-up sweeps for the latent-signal setup.
This notebook tests the residual nonlinear alpha value and varies signal strength versus noise in the cubic-signal dataset.


## 1. Imports

The path setup lets this notebook work when Jupyter starts either in the repository root or inside `experiment-notebooks/`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

GLOBAL_SEED = 123
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

P_DIM = 128
Q_DIM = 128

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / "contrastive_encoders").exists():
        module_root = candidate
        break
else:
    raise RuntimeError("Could not find the contrastive_encoders package folder.")

if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

PLOT_DIR = module_root / "reports" / "report-plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("module_root:", module_root)
print("plot output:", PLOT_DIR)


In [ ]:
import importlib

import contrastive_encoders
from contrastive_encoders import (
    architectures,
    data,
    experiments,
    losses,
    interpretability,
    metrics,
    plotting,
    regularization,
    training,
)

for module in [
    architectures,
    data,
    losses,
    interpretability,
    metrics,
    regularization,
    training,
    experiments,
    plotting,
]:
    importlib.reload(module)

importlib.reload(contrastive_encoders)

from contrastive_encoders import (
    TrainConfig,
    cubic_coordinate_probe_curve,
    friendly_results_table,
    generate_deterministic_relation_dataset,
    make_experiment_datasets,
    make_first_experiment_configs,
    make_model_spec_table,
    paper_contrastive_loss,
    plot_alpha_sweep_curve,
    plot_branch_ratio_history,
    plot_coordinate_probe_curves,
    plot_deterministic_snr_sweep,
    plot_metric_by_config,
    plot_signal_noise_sweep,
    plot_signal_recovery_by_config,
    plot_similarity_heatmap,
    plot_top5_retrieval_by_setting,
    plot_train_test_separation_by_setting,
    run_alpha_sweep,
    run_deterministic_relation_experiment,
    run_first_experiment,
    run_signal_noise_sweep,
    train_one_model_with_artifacts,
)


## 2. Mathematical objective

For embeddings `z_x = f_X(X)` and `z_y = f_Y(Y)`, define

```text
s_ij = <z_x_i, z_y_j>
```

The paper objective maximizes

```text
L = sum_i s_ii
    - sum_i log sum_j exp(s_ij)
    - sum_i log sum_j exp(s_ji)
```

The training code minimizes `-L / N` using `paper_contrastive_loss(...)`.

Important: this is **not** the standard cross-entropy InfoNCE replacement. By default it uses raw inner products, no loss-side normalization, and temperature `1.0`. The encoder architecture now has internal branch normalization so `alpha` controls the nonlinear correction, but the paper loss itself is unchanged.


## 3. Model comparison grid

The pure linear baseline is now a normalized linear encoder:

```text
g(u) = Normalize(G u)
```

The residual nonlinear models normalize both branches so `alpha` controls the nonlinear correction:

```text
g(u) = Normalize(G u) + alpha * Normalize(A1 sigma(A2 u + b))
```

The normalization layers use `BatchNorm1d(..., affine=False)`. The `affine=False` part is important because it prevents a learned scale parameter from secretly undoing `alpha`.

This also keeps the pure linear embedding scale controlled before the raw-dot-product loss sees it. So `alpha=0` means no nonlinear correction, while larger `alpha` lets the normalized MLP correction contribute more strongly.


In [ ]:
configs = make_first_experiment_configs(epochs=300)
model_table = make_model_spec_table(configs, p=P_DIM, q=Q_DIM)
display(model_table)


## 4. Alpha sweep

This tests a denser alpha grid:

```text
alpha = [0.0, 0.01, 0.03, 0.1, 0.3, 1.0]
```

The plot asks whether there is a useful middle range where the nonlinear correction is strong enough to help but not so strong that it overfits or destabilizes. The figure is saved to `reports/report-plots/`.


In [ ]:
alpha_sweep_results = run_alpha_sweep(
    seed=GLOBAL_SEED + 500,
    device=DEVICE,
    alphas=[0.0, 0.01, 0.03, 0.1, 0.3, 1.0],
    epochs=300,
    n_train=160,
    n_test=800,
    p=P_DIM,
    q=Q_DIM,
    signal_strength=2.0,
    noise_std=1.0,
)

plot_alpha_sweep_curve(
    alpha_sweep_results,
    metric="test_pair_separation",
    title="Alpha sweep: held-out true-pair separation",
    save_dir=PLOT_DIR,
)

display(
    friendly_results_table(
        alpha_sweep_results[
            [
                "setting",
                "config",
                "nonlinear_scale",
                "test_pair_separation",
                "shuffled_pair_separation",
                "test_top5_pair_match_accuracy",
                "x_signal_recovery",
                "x_related_signal_recovery",
                "y_signal_recovery",
                "x_probe_r2_z_x",
                "x_probe_r2_z_y",
                "y_probe_r2_z_y",
                "mean_nonlinear_to_linear_ratio",
            ]
        ].round(4)
    )
)


## 5. Signal strength / noise sweep

This sweep varies:

```text
signal_strength_values = [1.0, 2.0, 3.0, 5.0]
noise_std_values = [0.25, 0.5, 1.0, 2.0]
```

The x-axis is `signal_strength / noise_std`. If all models fail at low ratios and improve at high ratios, then the issue may be weak signal rather than the encoder architecture. The figure is saved to `reports/report-plots/`.


In [ ]:
signal_noise_configs = [
    TrainConfig(
        name="Linear encoder (alpha=0)",
        architecture="linear",
        hidden_dim=0,
        nonlinear_scale=0.0,
        epochs=300,
    ),
    TrainConfig(
        name="MLP nonlinear (alpha=0.10)",
        architecture="residual",
        hidden_dim=16,
        nonlinear_scale=0.10,
        epochs=300,
    ),
    TrainConfig(
        name="L2-regularized nonlinear (alpha=0.10)",
        architecture="residual",
        hidden_dim=16,
        nonlinear_scale=0.10,
        l2_nonlinear=1e-4,
        epochs=300,
    ),
]

signal_noise_results = run_signal_noise_sweep(
    configs=signal_noise_configs,
    seed=GLOBAL_SEED + 900,
    device=DEVICE,
    signal_strength_values=[1.0, 2.0, 3.0, 5.0],
    noise_std_values=[0.25, 0.5, 1.0, 2.0],
    setting="Cubic signal",
    n_train=160,
    n_test=800,
    p=P_DIM,
    q=Q_DIM,
)

plot_signal_noise_sweep(
    signal_noise_results,
    metric="test_pair_separation",
    title="Cubic signal: held-out separation versus signal/noise ratio",
    save_dir=PLOT_DIR,
)

display(
    friendly_results_table(
        signal_noise_results[
            [
                "setting",
                "config",
                "signal_strength",
                "noise_std",
                "signal_to_noise",
                "test_pair_separation",
                "shuffled_pair_separation",
                "test_top5_pair_match_accuracy",
                "x_signal_recovery",
                "x_related_signal_recovery",
                "y_signal_recovery",
                "x_probe_r2_z_x",
                "x_probe_r2_z_y",
                "y_probe_r2_z_y",
            ]
        ].round(4)
    )
)
